# 03 - Face-Based Sentiment Model

**Goal:** Train a ResNet18 classifier on cropped MSCTD faces to predict sentiment in {negative, neutral, positive}.

**Multiple-face / no-face policy** is realised in *evaluation*: per-face probabilities are aggregated to image-level via mean pooling (see notebook 06).

In [2]:
# Local CPU runtime setup for VS Code/Jupyter on this machine.
import os, sys
from pathlib import Path

PROJECT_NAME = "EEEM068-Human-Sentiment-Analysis"
LOCAL_PROJECT_ROOT = Path(r"C:/Users/hp/Desktop/CNN/EEEM068-Human-Sentiment-Analysis")
ENV_PROJECT_ROOT = "EEEM068_PROJECT_ROOT"

def _is_project_root(path: Path) -> bool:
    return (path / "src" / "config.py").is_file()

def _safe_resolve(path: Path):
    try:
        return path.expanduser().resolve()
    except Exception:
        return None

def _find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = []

    env_root = os.environ.get(ENV_PROJECT_ROOT)
    if env_root:
        candidates.append(Path(env_root))

    candidates.extend([cwd, *cwd.parents, LOCAL_PROJECT_ROOT])

    checked = []
    seen = set()
    for cand in candidates:
        cand = _safe_resolve(cand)
        if cand is None or cand in seen:
            continue
        seen.add(cand)
        checked.append(cand)
        if _is_project_root(cand):
            return cand

    checked_text = "\n".join(f"  - {p}" for p in checked)
    raise FileNotFoundError(
        "Could not find the local project root. Expected src/config.py.\n"
        f"Current working directory: {cwd}\n"
        f"Checked:\n{checked_text}\n\n"
        "Open this folder in VS Code and restart the notebook kernel:\n"
        "  C:/Users/hp/Desktop/CNN/EEEM068-Human-Sentiment-Analysis\n"
        f"Or set os.environ['{ENV_PROJECT_ROOT}'] to the exact local project path."
    )

ROOT = _find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print(f"Project root: {ROOT}")

import torch
from torch.utils.data import DataLoader
import pandas as pd

from src import config as C
from src.dataset import FaceDataset, load_face_metadata, class_weights_from_df
from src.transforms import standard_train_transform, standard_eval_transform
from src.models import build_face_model
from src.train import train_classifier, default_forward
from src.evaluate import evaluate_and_save
from src.utils import seed_everything, plot_training_curves, count_parameters, format_param_count

C.ensure_dirs()
seed_everything(42)
device = torch.device('cpu')
print('Device:', device)

Project root: C:\Users\hp\Desktop\CNN\EEEM068-Human-Sentiment-Analysis
Device: cpu


## 1. Load face metadata and build datasets

In [3]:
face_df = load_face_metadata(C.FACE_METADATA_CSV)
print('Total face crops:', len(face_df))
print(face_df['split'].value_counts())

train_tf = standard_train_transform(C.FACE_SIZE)
eval_tf = standard_eval_transform(C.FACE_SIZE)

train_ds = FaceDataset(face_df[face_df.split=='train'], transform=train_tf)
val_ds   = FaceDataset(face_df[face_df.split=='val'],   transform=eval_tf)
test_ds  = FaceDataset(face_df[face_df.split=='test'],  transform=eval_tf)

cfg = C.FACE_TRAIN
train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,  num_workers=0, pin_memory=False)
val_loader   = DataLoader(val_ds,   batch_size=cfg.batch_size, shuffle=False, num_workers=0, pin_memory=False)
test_loader  = DataLoader(test_ds,  batch_size=cfg.batch_size, shuffle=False, num_workers=0, pin_memory=False)
len(train_ds), len(val_ds), len(test_ds)

Total face crops: 43768
split
train    30801
test      6564
val       6403
Name: count, dtype: int64


(30801, 6403, 6564)

## 2. Build the model

In [4]:
model = build_face_model(num_classes=C.NUM_CLASSES, pretrained=True)
print('Trainable params:', format_param_count(count_parameters(model, True)))

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\hp/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 64.0MB/s]


Trainable params: 11.2M


## 3. Train

In [5]:
class_weights = class_weights_from_df(face_df[face_df.split=='train'])
print('class weights:', class_weights.tolist())

model, history = train_classifier(
    model, train_loader, val_loader,
    cfg=cfg, save_path=C.FACE_CKPT,
    class_weights=class_weights,
)

class weights: [1.003714919090271, 0.8757250308990479, 1.1603752374649048]


Ep1/15 train:   0%|          | 0/963 [00:00<?, ?it/s]

Ep1/15  val :   0%|          | 0/201 [00:00<?, ?it/s]

[Epoch 01] train_loss=1.1172 train_acc=0.358  val_loss=1.1061 val_acc=0.387  (2988.5s)


Ep2/15 train:   0%|          | 0/963 [00:00<?, ?it/s]

...

Ep5/15  val :   0%|          | 0/201 [00:00<?, ?it/s]

[Epoch 05] train_loss=1.0647 train_acc=0.421  val_loss=1.0935 val_acc=0.384  (1753.1s)


Ep6/15 train:   0%|          | 0/963 [00:00<?, ?it/s]

Ep6/15  val :   0%|          | 0/201 [00:00<?, ?it/s]

[Epoch 06] train_loss=1.0500 train_acc=0.438  val_loss=1.1182 val_acc=0.372  (2484.9s)
Early stopping at epoch 6 (no improvement for 3 epochs).


In [6]:
plot_training_curves(
    history.to_dict(),
    title='Face ResNet18 training',
    save_path=C.PLOTS_DIR/'face_model_training_curve.png',
)
plot_training_curves(
    history.to_dict(),
    title='Face ResNet18 training',
    save_path=C.REPORT_FIG_DIR/'face_model_training_curve.png',
)

## 4. Evaluate on the held-out test split (face-level)

In [7]:
metrics = evaluate_and_save(
    model, test_loader,
    forward_fn=default_forward,
    name='face_resnet18',
)
{k: v for k, v in metrics.items() if k in ('accuracy', 'macro_f1', 'weighted_f1')}

predict:   0%|          | 0/206 [00:00<?, ?it/s]

{'accuracy': 0.3820840950639854,
 'macro_f1': 0.3779928709096079,
 'weighted_f1': 0.3794826382607055}

## 5. Discussion to record

- The face model sees only the cropped face: it cannot use scene context. We therefore expect lower accuracy on `neutral`, where facial cues are subtle.
- Macro-F1 is reported alongside accuracy because the dataset is class-imbalanced.
- The `face_resnet18_predictions.csv` file (in `outputs/predictions/`) is consumed in notebook 06 to build the fusion features.

## 6. Observed results

Final test metrics for the face ResNet18 are saved in
`outputs/metrics/face_resnet18_metrics.json` and the confusion matrix
in `outputs/confusion_matrices/face_resnet18_cm.png`.

**Headline numbers** (face-level test set, n = 6,564):

| Metric | Value |
|---|---|
| Accuracy | **0.382** |
| Macro F1 | **0.378** |
| Weighted F1 | **0.379** |

**Per-class F1** (`outputs/metrics/face_resnet18_metrics.json`):
neutral **0.423**, negative **0.389**, positive **0.323**. The
ranking matches our prior: positive sentiment, which is most often
carried by a smile that the cropped 224x224 face captures well
in *some* frames but is ambiguous in many others, ends up as the
weakest class. Neutral is the easiest because the model can fall
back on "predict neutral" whenever the expression is flat - which is
also visible as the high recall (0.448) on neutral in the per-class
report.

**Improvement over the prior.** The majority-class baseline on the
matched test set sits at macro-F1 **0.183**, so the trained face
model more than doubles the macro-F1 of guessing the prior - a real
signal, but a clear ceiling: cropped faces alone discard the scene
context that disambiguates many neutral/negative frames. This is
exactly the gap that the full-image branch (Notebook 05) and the
fusion MLP (Notebook 06) are designed to close.

**Training behaviour.** The training-curve plot
(`outputs/plots/face_model_training_curve.png`) shows train loss
falling smoothly while validation loss flattens early - the
class-weighted cross-entropy plus the augmentation pipeline keep the
model from collapsing onto the neutral class, but the model is still
data-bound rather than capacity-bound at this size, which is why we
deliberately do **not** unfreeze the backbone here.